In [109]:
import pandas as pd

In [110]:
df = pd.read_csv("data/WA_Fn-UseC_-HR-Employee-Attrition.csv")

In [111]:
df.shape

(1470, 35)

In [112]:
df.head(3)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


In [113]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

In [114]:
category_columns = [
    "Attrition", "BusinessTravel", "Department",
    "EducationField", "Gender", "JobRole",
    "Over18", "OverTime", "MaritalStatus", "Education"
]

for col in category_columns:
    df[col] = df[col].astype("category")

In [115]:
df['is_attrition'] = (df['Attrition'] == 'Yes').astype(int)
df['is_overtime'] = (df['OverTime'] == 'Yes').astype(int)

In [116]:
df.nunique().rename("unique_count")

Age                           43
Attrition                      2
BusinessTravel                 3
DailyRate                    886
Department                     3
DistanceFromHome              29
Education                      5
EducationField                 6
EmployeeCount                  1
EmployeeNumber              1470
EnvironmentSatisfaction        4
Gender                         2
HourlyRate                    71
JobInvolvement                 4
JobLevel                       5
JobRole                        9
JobSatisfaction                4
MaritalStatus                  3
MonthlyIncome               1349
MonthlyRate                 1427
NumCompaniesWorked            10
Over18                         1
OverTime                       2
PercentSalaryHike             15
PerformanceRating              2
RelationshipSatisfaction       4
StandardHours                  1
StockOptionLevel               4
TotalWorkingYears             40
TrainingTimesLastYear          7
WorkLifeBa

In [117]:
for col in df.columns:
    if df[col].nunique() < 10:
        print(col, df[col].unique())

Attrition ['Yes', 'No']
Categories (2, str): ['No', 'Yes']
BusinessTravel ['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
Categories (3, str): ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
Department ['Sales', 'Research & Development', 'Human Resources']
Categories (3, str): ['Human Resources', 'Research & Development', 'Sales']
Education [2, 1, 4, 3, 5]
Categories (5, int64): [1, 2, 3, 4, 5]
EducationField ['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources']
Categories (6, str): ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
EmployeeCount [1]
EnvironmentSatisfaction [2 3 4 1]
Gender ['Female', 'Male']
Categories (2, str): ['Female', 'Male']
JobInvolvement [3 2 4 1]
JobLevel [2 1 3 4 5]
JobRole ['Sales Executive', 'Research Scientist', 'Laboratory Technician', 'Manufacturing Director', 'Healthcare Representative', 'Manager', 'Sales Representative', 'Research Director', 'Human Resources']
Categor

In [118]:
df_unique_counts = df.nunique().rename("unique_count")
non_unique_counts = df_unique_counts[df_unique_counts == 1].index.to_list()

In [119]:
df = df.drop(non_unique_counts, axis=1)

In [120]:
df.isna().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
YearsWithCurrManager        0
is_attrition                0
is_overtim

In [121]:
print(f"Полных дубликатов: {df.duplicated().sum()}")
print(f"Дубликатов EmployeeNumber: {df['EmployeeNumber'].duplicated().sum()}")
print(f"Всего пропусков в таблице: {df.isna().sum().sum()}")

checks = {
    "Возраст вне [18, 90]": ~df['Age'].between(18, 90),
    "Отрицательные/нулевые деньги и дистанция": (df[['DistanceFromHome', 'DailyRate', 'HourlyRate', 'MonthlyIncome', 'MonthlyRate', 'PercentSalaryHike']] <= 0).any(axis=1),
    "Отрицательный стаж или опыт": (df[['NumCompaniesWorked', 'TotalWorkingYears', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']] < 0).any(axis=1),
    "Стаж в компании > общего стажа": df['YearsAtCompany'] > df['TotalWorkingYears'],
    "Лет без повышения > стажа в компании": df['YearsSinceLastPromotion'] > df['YearsAtCompany'],
    "Начал работать раньше 16 лет": (df['Age'] - df['TotalWorkingYears']) < 16
}

for name, condition in checks.items():
    print(f"{name}: {condition.sum()}")

Полных дубликатов: 0
Дубликатов EmployeeNumber: 0
Всего пропусков в таблице: 0
Возраст вне [18, 90]: 0
Отрицательные/нулевые деньги и дистанция: 0
Отрицательный стаж или опыт: 0
Стаж в компании > общего стажа: 0
Лет без повышения > стажа в компании: 0
Начал работать раньше 16 лет: 0


In [122]:
df.to_parquet('data/clean.parquet', index=False)